# F1-score и другие метрики качества моделей

Этот ноутбук — практическое введение в метрики машинного обучения. Основное внимание уделено **F1-score для классификации**, но в конце есть краткий раздел про регрессию.

**Что вы узнаете:**

- почему одной доли правильных ответов (`accuracy`) часто недостаточно;
- что такое TP, FP, TN и FN;
- как связаны precision, recall и F1;
- чем отличаются micro-, macro- и weighted-F1;
- когда выбирать ROC-AUC, PR-AUC, MCC, log loss и другие метрики;
- как считать метрики с помощью `scikit-learn`.

> Все примеры используют синтетические данные, поэтому ноутбук можно запускать без скачивания файлов.

## 1. Зачем нужны разные метрики?

**Метрика качества** — это число, которое показывает, насколько хорошо модель решает задачу. Но слово «хорошо» зависит от цены ошибки.

Например, при поиске редкого заболевания:

- пропустить больного — опасно;
- ошибочно отправить здорового человека на дополнительное обследование — неприятно, но обычно менее критично.

Поэтому одна модель может быть лучше по одной метрике и хуже по другой. **Метрику выбирают исходя из задачи, баланса классов и стоимости ошибок.**

## 2. Матрица ошибок: основа метрик классификации

Рассмотрим бинарную классификацию: класс `1` — положительный, класс `0` — отрицательный.

|  | Модель предсказала `1` | Модель предсказала `0` |
|---|---:|---:|
| **На самом деле `1`** | **TP** — верно нашли положительный объект | **FN** — пропустили положительный объект |
| **На самом деле `0`** | **FP** — ложная тревога | **TN** — верно распознали отрицательный объект |

TP = true positive, FP = false positive, TN = true negative, FN = false negative.

Допустим, среди 100 пациентов 10 больны. Модель нашла 8 из них, а ещё у 12 здоровых пациентов ошибочно предсказала болезнь. Тогда:

- TP = 8, FN = 2;
- FP = 12, TN = 78.

In [ ]:
import numpy as np
import pandas as pd

# Простой пример: первые 10 объектов больны, остальные 90 здоровы.
y_true_small = np.array([1] * 10 + [0] * 90)
# Нашли 8 больных, пропустили 2; добавили 12 ложных тревог.
y_pred_small = np.array([1] * 8 + [0] * 2 + [1] * 12 + [0] * 78)

from sklearn.metrics import confusion_matrix, ConfusionMatrixDisplay
import matplotlib.pyplot as plt

cm_small = confusion_matrix(y_true_small, y_pred_small)
print('Матрица ошибок [[TN, FP], [FN, TP]]:')
print(cm_small)

fig, ax = plt.subplots(figsize=(5, 4))
ConfusionMatrixDisplay(cm_small, display_labels=['Здоров', 'Болен']).plot(
    ax=ax, cmap='Blues', colorbar=False, values_format='d'
)
ax.set_title('Матрица ошибок')
plt.show()

## 3. Accuracy: почему она может обманывать?

**Accuracy** — доля правильных предсказаний среди всех объектов:

$$\text{Accuracy}=\frac{TP+TN}{TP+TN+FP+FN}$$

В примере выше accuracy равна $(8+78)/100=0{,}86$, то есть 86%. Но если бы модель **всем** предсказала класс «здоров», accuracy была бы ещё выше — 90%. При этом она не нашла бы ни одного больного.

Это типичная проблема при **дисбалансе классов**: высокая accuracy не гарантирует, что модель полезна для редкого, но важного класса.

In [ ]:
from sklearn.metrics import accuracy_score

print('Accuracy исходной модели:', accuracy_score(y_true_small, y_pred_small))
print('Accuracy модели «всем 0»:  ', accuracy_score(y_true_small, np.zeros_like(y_true_small)))

## 4. Precision и recall

### Precision — точность положительных предсказаний

$$\text{Precision}=\frac{TP}{TP+FP}$$

Precision отвечает на вопрос: **«Среди объектов, которые модель назвала положительными, сколько действительно положительных?»**

Низкий precision означает много ложных тревог (FP). Он особенно важен, когда ложное срабатывание дорого: например, автоматическая блокировка пользователя.

### Recall — полнота

$$\text{Recall}=\frac{TP}{TP+FN}$$

Recall отвечает на вопрос: **«Какую долю всех реальных положительных объектов модель смогла найти?»**

Низкий recall означает много пропущенных положительных объектов (FN). Он важен, когда пропуск опасен: например, при поиске мошеннических операций или заболевания.

> В русскоязычной литературе precision иногда называют «точностью», а accuracy — «долей правильных ответов». Чтобы не путать термины, дальше используем английские названия.

In [ ]:
from sklearn.metrics import precision_score, recall_score

precision_small = precision_score(y_true_small, y_pred_small)
recall_small = recall_score(y_true_small, y_pred_small)

print(f'Precision: {precision_small:.3f} — из 20 предсказаний «болен» верны 8')
print(f'Recall:    {recall_small:.3f} — из 10 больных найдены 8')

## 5. Что такое F1-score?

**F1-score** — гармоническое среднее precision и recall:

$$F_1=2\cdot\frac{\text{Precision}\cdot\text{Recall}}{\text{Precision}+\text{Recall}}$$

Через элементы матрицы ошибок:

$$F_1=\frac{2TP}{2TP+FP+FN}$$

F1 изменяется от 0 до 1: **чем выше, тем лучше**. Метрика полезна, когда важен баланс между precision и recall, особенно при дисбалансе классов.

### Почему не обычное среднее?

Гармоническое среднее сильнее «наказывает» ситуацию, когда одна из двух величин близка к нулю.

Например, при precision = 1.0 и recall = 0.1:

- обычное среднее: $(1.0+0.1)/2=0.55$;
- F1: $2\cdot1.0\cdot0.1/(1.0+0.1)\approx0.182$.

F1 честнее показывает, что модель находит очень мало положительных объектов.

In [ ]:
from sklearn.metrics import f1_score

f1_small = f1_score(y_true_small, y_pred_small)
manual_f1 = 2 * precision_small * recall_small / (precision_small + recall_small)

print(f'F1 из sklearn:          {f1_small:.3f}')
print(f'F1 по формуле:          {manual_f1:.3f}')
print('F1 модели «всем 0»:    ', f1_score(y_true_small, np.zeros_like(y_true_small)))

### Важные свойства и ограничения F1

- F1 учитывает **TP, FP и FN**, но **не учитывает TN**.
- Для бинарной задачи результат зависит от того, какой класс выбран положительным (`pos_label`).
- F1 обычно считают **после выбора порога** предсказания класса, поэтому она зависит от порога.
- F1 одинаково штрафует FP и FN в формуле, хотя в реальной задаче их цена может различаться.
- Если важнее одна из составляющих, можно использовать $F_\beta$:

$$F_\beta=(1+\beta^2)\frac{\text{Precision}\cdot\text{Recall}}{\beta^2\cdot\text{Precision}+\text{Recall}}$$

При $\beta>1$ больший вес получает recall, при $\beta<1$ — precision.

In [ ]:
from sklearn.metrics import fbeta_score

print(f'F0.5 (больше веса у precision): {fbeta_score(y_true_small, y_pred_small, beta=0.5):.3f}')
print(f'F1 (равный баланс):             {fbeta_score(y_true_small, y_pred_small, beta=1):.3f}')
print(f'F2 (больше веса у recall):      {fbeta_score(y_true_small, y_pred_small, beta=2):.3f}')

## 6. Практический пример: обучим модель

Создадим несбалансированный набор данных: положительный класс встречается примерно в 10% объектов. Обучим логистическую регрессию и сравним метрики на тестовой выборке.

> Здесь модель нужна только для демонстрации метрик. В реальном проекте порог и гиперпараметры следует выбирать на валидационной выборке, а тестовую использовать для финальной оценки.

In [ ]:
from sklearn.datasets import make_classification
from sklearn.model_selection import train_test_split
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression

X, y = make_classification(
    n_samples=3000,
    n_features=15,
    n_informative=8,
    n_redundant=2,
    weights=[0.9, 0.1],
    class_sep=1.2,
    random_state=42
)

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.25, stratify=y, random_state=42
)

model = make_pipeline(
    StandardScaler(),
    LogisticRegression(max_iter=2000, random_state=42)
)
model.fit(X_train, y_train)

y_pred = model.predict(X_test)
y_proba = model.predict_proba(X_test)[:, 1]

print(f'Доля положительного класса в тесте: {y_test.mean():.3f}')
print(f'Размер тестовой выборки: {len(y_test)}')

In [ ]:
fig, ax = plt.subplots(figsize=(5, 4))
ConfusionMatrixDisplay.from_predictions(
    y_test, y_pred, ax=ax, cmap='Blues', colorbar=False, values_format='d'
)
ax.set_title('Матрица ошибок на тестовой выборке')
plt.show()

## 7. Другие метрики классификации

| Метрика | Что показывает | Когда особенно полезна | Важное ограничение |
|---|---|---|---|
| **Accuracy** | Доля всех верных ответов | Классы сбалансированы, ошибки примерно равноценны | Может быть обманчивой при дисбалансе |
| **Balanced accuracy** | Средний recall по классам | Несбалансированная классификация | Не показывает цену FP отдельно |
| **Precision** | Доля верных среди предсказанных `1` | Дорогие ложные тревоги | Не учитывает FN |
| **Recall / Sensitivity / TPR** | Доля найденных реальных `1` | Дорогие пропуски положительного класса | Не учитывает FP |
| **Specificity / TNR** | Доля верно распознанных реальных `0` | Важно не тревожить отрицательный класс | Не учитывает ошибки внутри положительного класса |
| **F1-score** | Баланс precision и recall | Важны оба типа ошибок FP и FN | Игнорирует TN, зависит от порога |
| **MCC** | Обобщённая корреляция предсказаний и истины | Бинарная задача, сильный дисбаланс | Менее интуитивна; диапазон от −1 до 1 |
| **ROC-AUC** | Качество ранжирования по всем порогам | Сравнение вероятностных классификаторов | При сильном дисбалансе может выглядеть слишком оптимистично |
| **PR-AUC / Average Precision** | Качество ранжирования для положительного класса | Редкий положительный класс | Зависит от доли положительного класса |
| **Log loss** | Качество и калиброванность вероятностей | Вероятности используются в решениях | Сильно штрафует уверенные ошибки |

**MCC** (Matthews correlation coefficient) учитывает все четыре клетки матрицы ошибок. Значение 1 — идеально, 0 — примерно на уровне случайного предсказания, −1 — полное противопоставление истине.

**ROC-AUC** можно интуитивно понимать как вероятность того, что случайно выбранный положительный объект получит оценку выше, чем случайно выбранный отрицательный.

**Log loss** применяется к вероятностям, а не только к итоговым классам: чем увереннее неправильное предсказание, тем больше штраф.

In [ ]:
from sklearn.metrics import (
    accuracy_score, balanced_accuracy_score, precision_score, recall_score,
    f1_score, matthews_corrcoef, roc_auc_score, average_precision_score,
    log_loss
)

# specificity = TN / (TN + FP)
tn, fp, fn, tp = confusion_matrix(y_test, y_pred).ravel()
specificity = tn / (tn + fp)

results = pd.DataFrame({
    'Метрика': [
        'Accuracy', 'Balanced accuracy', 'Precision', 'Recall', 'Specificity',
        'F1', 'MCC', 'ROC-AUC', 'Average Precision (PR)', 'Log loss'
    ],
    'Значение': [
        accuracy_score(y_test, y_pred),
        balanced_accuracy_score(y_test, y_pred),
        precision_score(y_test, y_pred),
        recall_score(y_test, y_pred),
        specificity,
        f1_score(y_test, y_pred),
        matthews_corrcoef(y_test, y_pred),
        roc_auc_score(y_test, y_proba),
        average_precision_score(y_test, y_proba),
        log_loss(y_test, y_proba)
    ]
})

# Log loss — исключение: для него меньше означает лучше.
results.style.format({'Значение': '{:.3f}'})

Обратите внимание: у log loss направление другое — **меньше = лучше**. У остальных перечисленных выше метрик обычно **больше = лучше**, а у MCC нулевое значение соответствует примерно случайному уровню.

In [ ]:
from sklearn.metrics import classification_report

print(classification_report(y_test, y_pred, target_names=['Класс 0', 'Класс 1'], digits=3))

## 8. F1 и порог классификации

Модель часто выдаёт вероятность $p$, а класс получают сравнением с порогом $t$:

$$\hat y = \mathbb{1}[p\ge t]$$

Стандартный порог 0.5 не обязан быть оптимальным:

- при повышении порога обычно становится меньше FP, но больше FN;
- при снижении порога обычно растёт recall, но precision может падать.

Посмотрим, как precision, recall и F1 меняются вместе с порогом. **В реальном проекте лучший порог выбирают на валидационной, а не на тестовой выборке.**

In [ ]:
from sklearn.metrics import precision_recall_curve

precision_curve, recall_curve, thresholds = precision_recall_curve(y_test, y_proba)
# У precision и recall на один элемент больше, поэтому отбрасываем последний.
f1_curve = 2 * precision_curve[:-1] * recall_curve[:-1] / (
    precision_curve[:-1] + recall_curve[:-1] + 1e-12
)

fig, ax = plt.subplots(figsize=(9, 5))
ax.plot(thresholds, precision_curve[:-1], label='Precision')
ax.plot(thresholds, recall_curve[:-1], label='Recall')
ax.plot(thresholds, f1_curve, label='F1', linewidth=2.5)
ax.axvline(0.5, color='gray', linestyle='--', alpha=0.7, label='Порог 0.5')
ax.set(xlabel='Порог', ylabel='Значение метрики', title='Зависимость метрик от порога', ylim=(0, 1.02))
ax.legend()
ax.grid(alpha=0.2)
plt.show()

## 9. Micro, macro и weighted F1

Когда классов больше двух, нужно решить, как объединить качество по классам.

Пусть есть классы A, B и C:

- **Macro-F1** — посчитать F1 отдельно для каждого класса и взять обычное среднее. Все классы равноправны, включая редкие.
- **Weighted-F1** — посчитать F1 по классам и усреднить с весами, пропорциональными числу объектов класса (`support`). Частые классы влияют сильнее.
- **Micro-F1** — сложить TP, FP и FN по всем классам и только затем посчитать F1. Каждый объект вносит примерно равный вклад.

Для обычной **одноклассовой разметки каждого объекта** (single-label multiclass) micro-F1 численно равен accuracy. В multilabel-задаче, где у объекта может быть несколько меток, это равенство уже не обязано выполняться.

In [ ]:
# Три класса; класс 2 встречается реже остальных.
y_true_multi = np.array([0, 0, 0, 0, 1, 1, 1, 2, 2, 2])
y_pred_multi = np.array([0, 0, 1, 0, 1, 1, 0, 2, 1, 0])

f1_values = {
    'Micro-F1': f1_score(y_true_multi, y_pred_multi, average='micro'),
    'Macro-F1': f1_score(y_true_multi, y_pred_multi, average='macro'),
    'Weighted-F1': f1_score(y_true_multi, y_pred_multi, average='weighted')
}
print(pd.Series(f1_values).round(3).to_string())
print()
print('Подробный отчёт:')
print(classification_report(y_true_multi, y_pred_multi, digits=3, zero_division=0))

### Какой вариант усреднения выбрать?

- Нужно одинаково хорошо распознавать **все классы**, даже редкие → **macro-F1**.
- Нужен общий результат с учётом частоты классов → **weighted-F1**.
- Важен вклад каждого отдельного объекта/метки → часто смотрят **micro-F1**.

Полезно выводить не только сводную метрику, но и значения по каждому классу: усреднение может скрывать слабый класс.

## 10. ROC-AUC и PR-AUC: метрики до выбора порога

Precision, recall и F1 обычно оценивают конкретный набор предсказанных классов. ROC-AUC и PR-AUC оценивают **ранжирование объектов по вероятности** сразу по многим порогам.

- **ROC** строится по TPR (recall) и FPR = FP / (FP + TN).
- **PR-кривая** строится по precision и recall и сфокусирована на положительном классе.

При редком положительном классе PR-AUC / Average Precision часто информативнее, чем ROC-AUC. Однако значения average precision нельзя напрямую сравнивать между наборами данных с сильно разной долей положительного класса.

In [ ]:
from sklearn.metrics import RocCurveDisplay, PrecisionRecallDisplay

fig, axes = plt.subplots(1, 2, figsize=(12, 5))
RocCurveDisplay.from_predictions(y_test, y_proba, ax=axes[0])
axes[0].set_title('ROC-кривая')
PrecisionRecallDisplay.from_predictions(y_test, y_proba, ax=axes[1])
axes[1].set_title('Precision–Recall кривая')
plt.tight_layout()
plt.show()

## 11. А что в задачах регрессии?

В регрессии модель предсказывает число, поэтому F1 не применяется. Основные метрики:

| Метрика | Формула / смысл | Особенности |
|---|---|---|
| **MAE** | Среднее $|y-\hat y|$ | Понятные единицы, устойчивее к выбросам, чем MSE |
| **MSE** | Среднее $(y-\hat y)^2$ | Сильнее штрафует большие ошибки |
| **RMSE** | $\sqrt{MSE}$ | Как MSE, но в исходных единицах |
| **R²** | Доля объяснённой дисперсии | 1 — идеально; может быть отрицательным на новых данных |
| **MAPE** | Средняя абсолютная ошибка в процентах | Не подходит при нулевых $y$, нестабилен около нуля |

Для всех метрик ошибки **меньше = лучше**. Для R² обычно **больше = лучше**.

In [ ]:
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

# Небольшой иллюстративный пример регрессии.
y_true_reg = np.array([100., 150., 200., 250., 300.])
y_pred_reg = np.array([110., 140., 220., 230., 330.])

mae = mean_absolute_error(y_true_reg, y_pred_reg)
mse = mean_squared_error(y_true_reg, y_pred_reg)
rmse = np.sqrt(mse)
r2 = r2_score(y_true_reg, y_pred_reg)
mape = np.mean(np.abs((y_true_reg - y_pred_reg) / y_true_reg)) * 100

print(f'MAE:  {mae:.2f}')
print(f'MSE:  {mse:.2f}')
print(f'RMSE: {rmse:.2f}')
print(f'R²:   {r2:.3f}')
print(f'MAPE: {mape:.2f}%')

## 12. Метод `.score()` — это не всегда F1

В `scikit-learn` у многих моделей есть метод `model.score(X, y)`. Его смысл зависит от типа модели:

- у классификаторов по умолчанию это обычно **accuracy**;
- у регрессоров по умолчанию это обычно **R²**.

Поэтому не стоит считать, что «score» — универсальная метрика. Для конкретной задачи лучше явно вызывать `f1_score`, `roc_auc_score`, `mean_absolute_error` и т. д.

In [ ]:
print('model.score для нашего классификатора:', model.score(X_test, y_test))
print('accuracy_score:                       ', accuracy_score(y_test, y_pred))
print('f1_score:                             ', f1_score(y_test, y_pred))

## 13. Краткая шпаргалка по выбору

1. **Сбалансированные классы, одинаковая цена ошибок** → начните с accuracy, но смотрите матрицу ошибок.
2. **Нельзя пропускать положительный класс** → recall, затем оцените precision и цену FP.
3. **Нельзя допускать ложные тревоги** → precision, затем оцените recall и цену FN.
4. **Нужен баланс precision и recall** → F1; при неравной цене ошибок рассмотрите $F_\beta$.
5. **Много классов, редкие классы тоже важны** → macro-F1 и отчёт по каждому классу.
6. **Нужно оценить ранжирование, порог ещё не выбран** → ROC-AUC; при редком положительном классе также PR-AUC / Average Precision.
7. **Важно качество самих вероятностей** → log loss и калибровка вероятностей.
8. **Регрессия** → MAE или RMSE в единицах целевой переменной; R² как дополнительная сводная метрика.

> Хорошая практика — заранее выбрать одну основную метрику для сравнения моделей и несколько дополнительных для проверки рисков.

## 14. Частые ошибки

- **Смотреть только на accuracy** при сильном дисбалансе классов.
- **Подбирать порог на тестовой выборке.** Это приводит к утечке информации и слишком оптимистичной оценке.
- **Сравнивать F1 между разными наборами данных** без учёта баланса классов и постановки задачи.
- **Перепутать положительный класс.** В бинарной F1 выбор `pos_label` меняет результат.
- **Усреднять без проверки отдельных классов.** Высокая weighted-F1 может скрывать плохой редкий класс.
- **Считать ROC-AUC мерой качества вероятностей.** AUC оценивает ранжирование, но не гарантирует хорошую калибровку.
- **Использовать MAPE при нулевых значениях** целевой переменной.

## 15. Упражнения

1. В примере с пациентами уменьшите число FP с 12 до 4. Как изменятся precision, recall и F1? Какая метрика не изменится?
2. Постройте предсказания модели при порогах 0.2, 0.5 и 0.8. Сравните precision, recall, F1 и матрицы ошибок.
3. Замените обычную F1 на F2 при разных порогах. Почему оптимальный порог может измениться?
4. В многоклассовом примере улучшите предсказания только для частого класса 0. Какая из метрик — macro-F1 или weighted-F1 — отреагирует сильнее?
5. Добавьте к регрессионным предсказаниям один большой выброс. Сравните реакцию MAE и RMSE.

In [ ]:
# Заготовка для упражнения 2: раскомментируйте и дополните код.
# for threshold in [0.2, 0.5, 0.8]:
#     pred = (y_proba >= threshold).astype(int)
#     print(f'Порог: {threshold}')
#     print('Precision:', precision_score(y_test, pred))
#     print('Recall:   ', recall_score(y_test, pred))
#     print('F1:       ', f1_score(y_test, pred))

## Итог

**F1-score** — это гармоническое среднее precision и recall. Она помогает оценить баланс между ложными тревогами и пропусками положительного класса, но не является универсальной метрикой.

Перед выбором метрики ответьте на три вопроса:

1. Какая ошибка опаснее — FP или FN?
2. Насколько сбалансированы классы?
3. Нужно оценивать итоговые классы, ранжирование или сами вероятности?

Правильная метрика — та, которая отражает реальную цель и стоимость ошибок в вашей задаче.